Paso 1 - Crear tabla particionada

Creamos una nueva tabla particionada utilizando la columna

In [ ]:
sql_create_partition = """
CREATE OR REPLACE TABLE retail_dw.ventas_particionadas
(
    id_venta INT64,
    fecha_venta DATE,
    id_cliente INT64,
    id_producto INT64,
    sucursal STRING,
    monto FLOAT64
)
PARTITION BY fecha_venta
AS
SELECT *
FROM retail_dw.ventas;
"""

job = client.query(sql_create_partition)
job.result()

print("✅ Tabla particionada creada correctamente")

Paso 2 - Verificar tabla particionada

In [ ]:
sql_check_partition = """
SELECT
    table_name,
    creation_time
FROM retail_dw.INFORMATION_SCHEMA.TABLES
WHERE table_name='ventas_particionadas';
"""

df_partition = client.query(sql_check_partition).to_dataframe()

print(df_partition)

Paso 3 - Consulta sobre tabla original

Ejecutamos una consulta filtrando por fecha.

In [ ]:
import time

sql_original = """
SELECT
    SUM(monto) total_ventas
FROM retail_dw.ventas
WHERE fecha_venta BETWEEN '2025-01-01' AND '2025-03-31';
"""

inicio = time.time()

job_original = client.query(sql_original)
job_original.result()

tiempo_original = time.time() - inicio

print(f"Tiempo tabla original: {tiempo_original:.3f} segundos")

Paso 4 - Consulta sobre tabla particionada

In [ ]:
sql_partition = """
SELECT
    SUM(monto) total_ventas
FROM retail_dw.ventas_particionadas
WHERE fecha_venta BETWEEN '2025-01-01' AND '2025-03-31';
"""

inicio = time.time()

job_partition = client.query(sql_partition)
job_partition.result()

tiempo_partition = time.time() - inicio

print(f"Tiempo tabla particionada: {tiempo_partition:.3f} segundos")

Paso 5 - Comparar rendimiento

In [ ]:
bytes_original = job_original.total_bytes_processed or 0
bytes_partition = job_partition.total_bytes_processed or 0

print("="*50)

print(f"⏱ Tiempo Original:      {tiempo_original:.3f}s")
print(f"⏱ Tiempo Particionada:  {tiempo_partition:.3f}s")

print("="*50)

print(f"📦 Datos Original:      {bytes_original/1e6:.2f} MB")
print(f"📦 Datos Particionada:  {bytes_partition/1e6:.2f} MB")

Paso 1 - Crear tabla clusterizada

In [ ]:
sql_create_cluster = """
CREATE OR REPLACE TABLE retail_dw.ventas_cluster
(
    id_venta INT64,
    fecha_venta DATE,
    id_cliente INT64,
    id_producto INT64,
    sucursal STRING,
    monto FLOAT64
)
PARTITION BY fecha_venta
CLUSTER BY sucursal
AS
SELECT *
FROM retail_dw.ventas;
"""

job = client.query(sql_create_cluster)
job.result()

print("✅ Tabla clusterizada creada correctamente")

Paso 2 - Verificar tabla clusterizada

In [ ]:
sql_check_cluster = """
SELECT
    table_name,
    creation_time
FROM retail_dw.INFORMATION_SCHEMA.TABLES
WHERE table_name='ventas_cluster';
"""

df_cluster = client.query(sql_check_cluster).to_dataframe()

print(df_cluster)

Paso 3 - Consulta sobre tabla particionada

In [ ]:
import time

sql_partition = """
SELECT
    SUM(monto)
FROM retail_dw.ventas_particionadas
WHERE sucursal='San Salvador';
"""

inicio = time.time()

job_partition = client.query(sql_partition)
job_partition.result()

tiempo_partition = time.time() - inicio

In [ ]:
Paso 4 - Consulta sobre tabla clusterizada

In [ ]:
sql_cluster = """
SELECT
    SUM(monto)
FROM retail_dw.ventas_cluster
WHERE sucursal='San Salvador';
"""

inicio = time.time()

job_cluster = client.query(sql_cluster)
job_cluster.result()

tiempo_cluster = time.time() - inicio

In [ ]:
Paso 5 - Comparar rendimiento

In [ ]:
bytes_partition = job_partition.total_bytes_processed or 0
bytes_cluster = job_cluster.total_bytes_processed or 0

print("="*50)

print(f"⏱ Tiempo Particionada: {tiempo_partition:.3f}s")
print(f"⏱ Tiempo Clusterizada: {tiempo_cluster:.3f}s")

print("="*50)

print(f"📦 Datos Particionada: {bytes_partition/1e6:.2f} MB")
print(f"📦 Datos Clusterizada: {bytes_cluster/1e6:.2f} MB")